In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.simplefilter("ignore")

In [2]:
df_train = pd.read_csv("/kaggle/input/playground-series-s5e5/train.csv")
df_test = pd.read_csv("/kaggle/input/playground-series-s5e5/test.csv")

# Data Preprocessing

In [3]:
num_vars = df_train.drop(columns=['id','Calories']).select_dtypes(include=['int64', 'float64']).columns
cat_vars = ['Sex']

In [4]:
df_train['Sex'] = df_train['Sex'].astype('category')
df_test['Sex'] = df_test['Sex'].astype('category')

# Feature Engineering

In [5]:
from sklearn.preprocessing import StandardScaler

def make_features(df):
    df_temp = df.copy()

    df_temp.drop(columns=['id'], inplace=True)
    # dummie encoding
    df_temp = pd.get_dummies(df_temp, columns=['Sex'])
    
    # Log Transformations
    df_temp['log_heart_rate'] = np.log(df_temp['Heart_Rate'])
    df_temp['log_duration'] = np.log(df_temp['Duration'])

    df_temp['exp_body_temp'] = np.exp(df_temp['Body_Temp'])

    # standardization
    features_to_scale = df_temp.drop(columns=['Calories']).select_dtypes(include=['int64', 'float64']).columns
    scaler = StandardScaler()
    df_temp[features_to_scale] = scaler.fit_transform(df_temp[features_to_scale])
    
    return df_temp

# for predicting log of outcome rather than just outcome
def make_features_log(df):
    df_temp = df.copy()

    df_temp.drop(columns=['id'], inplace=True)
    # dummie encoding
    df_temp = pd.get_dummies(df_temp, columns=['Sex'])
    
    # Log Transformations
    df_temp['log_heart_rate'] = np.log(df_temp['Heart_Rate'])
    df_temp['log_duration'] = np.log(df_temp['Duration'])
    df_temp['log_calories'] = np.log(df_temp['Calories'])

    df_temp['exp_body_temp'] = np.exp(df_temp['Body_Temp'])

    # standardization
    features_to_scale = df_temp.drop(columns=['log_calories','Calories']).select_dtypes(include=['int64', 'float64']).columns
    scaler = StandardScaler()
    df_temp[features_to_scale] = scaler.fit_transform(df_temp[features_to_scale])
    
    return df_temp

df_train1 = make_features(df_train)
df_train2 = make_features_log(df_train)

# Model

## XGB Baseline

In [6]:
SEED = 7

In [7]:
import xgboost as xgb
from sklearn.metrics import mean_squared_log_error
from sklearn.model_selection import train_test_split

In [8]:
# without the fancy features
X = df_train.drop(columns=['id', 'Calories'])
X = pd.get_dummies(X, columns=['Sex'])
y = df_train['Calories']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=SEED)

# with the fancy features
X1 = df_train1.drop(columns=['Calories'])
y1 = df_train1['Calories']
X_train1, X_val1, _, _ = train_test_split(X1, y1, test_size=0.3, random_state=SEED)

# with log calories
X2 = df_train2.drop(columns=['Calories', 'log_calories'])
y2 = df_train2['log_calories']
X_train2, X_val2, y_train2, y_val2 = train_test_split(X2, y2, test_size=0.3, random_state=SEED)

In [9]:
# baseline without new features
xgb_baseline = xgb.XGBRegressor(enable_calegorical=True)
xgb_baseline.fit(X_train, y_train)

y_val_pred = xgb_baseline.predict(X_val)
y_val_pred = np.maximum(y_val_pred, 0)
score = mean_squared_log_error(y_val_pred, y_val)
print(f'XGB Baseline Score: {score}')

XGB Baseline Score: 0.004272607917848008


In [10]:
# baseline with new features
xgb_baseline1 = xgb.XGBRegressor(enable_calegorical=True)
xgb_baseline1.fit(X_train1, y_train)

y_val_pred = xgb_baseline1.predict(X_val1)
y_val_pred = np.maximum(y_val_pred, 0)
score = mean_squared_log_error(y_val_pred, y_val)
print(f'XGB Baseline Score With new Features: {score}')

XGB Baseline Score With new Features: 0.004272607917848008


In [11]:
# baseline with log calories
xgb_baseline2 = xgb.XGBRegressor(enable_calegorical=True)
xgb_baseline2.fit(X_train2, y_train2)

y_val_pred = xgb_baseline2.predict(X_val2)
y_val_pred = np.exp(y_val_pred)
y_val_pred = np.maximum(y_val_pred, 0)
score = mean_squared_log_error(y_val_pred, y_val)
print(f'XGB Baseline Score With log Calories: {score}')

XGB Baseline Score With log Calories: 0.0038988490010866375


## Big Tuna

In [12]:
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold

In [13]:
def msle_eval(y_pred, dtrain):
    y_true = dtrain.get_label()
    y_pred = np.maximum(y_pred, 0)
    loss = mean_squared_log_error(y_true, y_pred)
    return 'MSLE', loss

# Function to run k-fold cross-validation with XGBoost and MSLE
def xgb_cv_msle(X, y, params, num_folds=5, early_stopping_rounds=50, debug=False):
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=SEED)
    fold_scores = []
    
    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        dtrain = xgb.DMatrix(X_train, label=y_train)
        dval = xgb.DMatrix(X_val, label=y_val)
        
        model = xgb.train(
            params,
            dtrain,
            evals=[(dval, 'val')],
            feval=msle_eval,
            early_stopping_rounds=early_stopping_rounds,
            verbose_eval=False
        )
        
        # Get predictions and ensure they're positive
        y_val_pred = model.predict(dval)
        y_val_pred = np.maximum(0, y_val_pred)
        score = mean_squared_log_error(y_val, y_val_pred)
        if debug == True:
            print(score)
        fold_scores.append(score)
        
    return fold_scores

# taking into account the log calories
def xgb_cv_msle_log(X, y_log, y, params, num_folds=5, early_stopping_rounds=50, debug=False):
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=SEED)
    fold_scores = []
    
    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train_log, y_val_log = y_log.iloc[train_idx], y_log.iloc[val_idx] # train using log
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        dtrain = xgb.DMatrix(X_train, label=y_train_log)
        dval_log = xgb.DMatrix(X_val, label=y_val_log)
        dval = xgb.DMatrix(X_val, label=y_val)
        
        model = xgb.train(
            params,
            dtrain,
            evals=[(dval_log, 'val')],
            feval=msle_eval,
            early_stopping_rounds=early_stopping_rounds,
            verbose_eval=False
        )
        
        # Get predictions and ensure they're positive
        y_val_pred = model.predict(dval_log)
        y_val_pred = np.exp(y_val_pred)
        y_val_pred = np.maximum(0, y_val_pred)
        score = mean_squared_log_error(y_val, y_val_pred)

        if debug == True:
            print(score)

        fold_scores.append(score)
        
    return fold_scores

In [14]:
def objective(trial):
    params = {
        # "objective": "reg:squarederror",
        # "eval_metric": "msle",
        "tree_method": "gpu_hist",
        "predictor": "gpu_predictor",
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.1, step=0.01),
        "max_depth": trial.suggest_int("max_depth", 5, 20),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0, step=0.1),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0, step=0.1),
        "max_bin": trial.suggest_int("max_bin", 256, 2048),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "lambda": trial.suggest_float("lambda", 1e-3, 10.0, log=True),
        "alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
        "grow_policy": trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"]),
        "n_estimators": trial.suggest_int("n_estimators", 50, 2000),
        "random_state": SEED
    }
    
    score = np.mean(xgb_cv_msle_log(X=X2, y_log=y2, y=y, params=params, debug=False))
    # score = np.mean(xgb_cv_msle(X=X, y=y, params=params, debug=True))
    return score

In [15]:
# %%time
# study = optuna.create_study(direction='minimize',
#                             sampler = optuna.samplers.RandomSampler(seed=SEED),
#                             study_name = "BIG BLUE FIN TUNA!!")
# study.optimize(objective, n_trials=100, show_progress_bar=True, )

In [16]:
# best_params = study.best_params
# print(f'Best Trial Params: {best_params}')

# print(f'Best Trial Value: {study.best_trial.value}')

In [17]:
# for saving versions
best_params = {'learning_rate': 0.1, 'max_depth': 18, 'subsample': 1.0, 'colsample_bytree': 1.0, 'max_bin': 1078, 'min_child_weight': 9, 'gamma': 0.991787479086812, 'lambda': 0.4896645151400931, 'alpha': 0.008889500949028779, 'grow_policy': 'lossguide', 'n_estimators': 879}


# Submission

In [18]:
def make_features_test(df):
    df_temp = df.copy()

    df_temp.drop(columns=['id'], inplace=True)
    # dummie encoding
    df_temp = pd.get_dummies(df_temp, columns=['Sex'])
    
    # Log Transformations
    df_temp['log_heart_rate'] = np.log(df_temp['Heart_Rate'])
    df_temp['log_duration'] = np.log(df_temp['Duration'])
    df_temp['exp_body_temp'] = np.exp(df_temp['Body_Temp'])

    # standardization
    features_to_scale = df_temp.select_dtypes(include=['int64', 'float64']).columns
    scaler = StandardScaler()
    df_temp[features_to_scale] = scaler.fit_transform(df_temp[features_to_scale])
    
    return df_temp

In [19]:
best_model = xgb.XGBRegressor(**best_params)
best_model.fit(X2, y2)

df_test= make_features_test(df_test)

y_test_pred = best_model.predict(df_test)
y_test_pred = np.exp(y_test_pred)
y_test_pred = np.maximum(y_test_pred, 0)

submission = pd.read_csv("/kaggle/input/playground-series-s5e5/sample_submission.csv")
submission['Calories'] = y_test_pred
submission.to_csv('submission.csv', index=False)
submission.head()

,id,Calories
0,750000,27.416031
1,750001,107.344368
2,750002,88.585464
3,750003,126.696320
4,750004,75.786232


#